https://drive.google.com/drive/folders/1p-RicE4Sx0Q2rqX5BbP9pl_9nSxYcbqn

In [6]:
from google.colab import files
uploaded=files.upload()

Saving machine_configurations.json to machine_configurations.json
Saving plant_master.csv to plant_master.csv
Saving manufacturing_production_120k.csv to manufacturing_production_120k.csv
Saving manufacturing_production_extra_12k.csv to manufacturing_production_extra_12k.csv
Saving machine_master.csv to machine_master.csv


In [3]:
!pip install pyspark

In [4]:
from pyspark.sql import SparkSession
spark=SparkSession.builder.appName("Tasks_23rd_Oct").getOrCreate()

**1. Load the Data in PySpark**

In [7]:
df=spark.read.csv("manufacturing_production_120k.csv",header=True,inferSchema=True)

In [8]:
df.show(20)

+-------------+---------------+--------+----------+-----------+-------+------------+--------------+-------------+--------------+---------------+----------------+----------+-------------+--------------+--------------+----------------+----------+
|production_id|production_date|plant_id|machine_id|operator_id|  shift|product_code|product_family|units_planned|units_produced|defective_units|downtime_minutes|energy_kwh|temperature_c|vibration_mm_s|quality_status|maintenance_flag|supervisor|
+-------------+---------------+--------+----------+-----------+-------+------------+--------------+-------------+--------------+---------------+----------------+----------+-------------+--------------+--------------+----------------+----------+
|    PR0000001|     2026-08-07|    PL03|     MC011|     OP0275|Evening|     MTR-100|        Motors|         1060|           993|             30|              34|    388.11|         61.4|          3.88|        Rework|              No|    Charan|
|    PR0000002|     

In [9]:
df.printSchema()

root
 |-- production_id: string (nullable = true)
 |-- production_date: date (nullable = true)
 |-- plant_id: string (nullable = true)
 |-- machine_id: string (nullable = true)
 |-- operator_id: string (nullable = true)
 |-- shift: string (nullable = true)
 |-- product_code: string (nullable = true)
 |-- product_family: string (nullable = true)
 |-- units_planned: integer (nullable = true)
 |-- units_produced: integer (nullable = true)
 |-- defective_units: integer (nullable = true)
 |-- downtime_minutes: integer (nullable = true)
 |-- energy_kwh: double (nullable = true)
 |-- temperature_c: double (nullable = true)
 |-- vibration_mm_s: double (nullable = true)
 |-- quality_status: string (nullable = true)
 |-- maintenance_flag: string (nullable = true)
 |-- supervisor: string (nullable = true)



In [10]:
print(df.columns)

['production_id', 'production_date', 'plant_id', 'machine_id', 'operator_id', 'shift', 'product_code', 'product_family', 'units_planned', 'units_produced', 'defective_units', 'downtime_minutes', 'energy_kwh', 'temperature_c', 'vibration_mm_s', 'quality_status', 'maintenance_flag', 'supervisor']


In [11]:
print(df.select("*").count())

120000


In [12]:
df.select("production_id","plant_id","machine_id","product_family","units_produced").show()

+-------------+--------+----------+--------------+--------------+
|production_id|plant_id|machine_id|product_family|units_produced|
+-------------+--------+----------+--------------+--------------+
|    PR0000001|    PL03|     MC011|        Motors|           993|
|    PR0000002|    PL04|     MC019|   Controllers|           836|
|    PR0000003|    PL01|     MC002|        Motors|           380|
|    PR0000004|    PL03|     MC013|         Pumps|           720|
|    PR0000005|    PL04|     MC020|         Pumps|           296|
|    PR0000006|    PL01|     MC002|        Motors|           795|
|    PR0000007|    PL02|     MC007|        Valves|           955|
|    PR0000008|    PL02|     MC009|        Valves|           895|
|    PR0000009|    PL03|     MC015|   Controllers|           289|
|    PR0000010|    PL02|     MC006|   Controllers|           527|
|    PR0000011|    PL03|     MC012|         Pumps|           512|
|    PR0000012|    PL06|     MC030|   Controllers|           968|
|    PR000

In [13]:
print(df.dtypes)

[('production_id', 'string'), ('production_date', 'date'), ('plant_id', 'string'), ('machine_id', 'string'), ('operator_id', 'string'), ('shift', 'string'), ('product_code', 'string'), ('product_family', 'string'), ('units_planned', 'int'), ('units_produced', 'int'), ('defective_units', 'int'), ('downtime_minutes', 'int'), ('energy_kwh', 'double'), ('temperature_c', 'double'), ('vibration_mm_s', 'double'), ('quality_status', 'string'), ('maintenance_flag', 'string'), ('supervisor', 'string')]


In [14]:
sample_df=df.sample(fraction=0.5)
sample_df.show(10)

+-------------+---------------+--------+----------+-----------+-------+------------+--------------+-------------+--------------+---------------+----------------+----------+-------------+--------------+--------------+----------------+----------+
|production_id|production_date|plant_id|machine_id|operator_id|  shift|product_code|product_family|units_planned|units_produced|defective_units|downtime_minutes|energy_kwh|temperature_c|vibration_mm_s|quality_status|maintenance_flag|supervisor|
+-------------+---------------+--------+----------+-----------+-------+------------+--------------+-------------+--------------+---------------+----------------+----------+-------------+--------------+--------------+----------------+----------+
|    PR0000001|     2026-08-07|    PL03|     MC011|     OP0275|Evening|     MTR-100|        Motors|         1060|           993|             30|              34|    388.11|         61.4|          3.88|        Rework|              No|    Charan|
|    PR0000005|     

**2. Creating DataFrames**

In [15]:
data=[(1,"Plant A","Chennai"),(2,"Plant B","Hyderabad"),(3,"Plant C","Coimbatore")]
columns=["plant_id","plant_name","city"]
plant_small_df=spark.createDataFrame(data,columns)
plant_small_df.show()

+--------+----------+----------+
|plant_id|plant_name|      city|
+--------+----------+----------+
|       1|   Plant A|   Chennai|
|       2|   Plant B| Hyderabad|
|       3|   Plant C|Coimbatore|
+--------+----------+----------+



In [16]:
from pyspark.sql.types import *

In [17]:
schema=StructType([
    StructField("plant_id",IntegerType(),True),
    StructField("plant_name",StringType(),True),
    StructField("city",StringType(),True)
])

In [18]:
plant_struct_df=spark.createDataFrame(data,schema)
plant_struct_df.show()

+--------+----------+----------+
|plant_id|plant_name|      city|
+--------+----------+----------+
|       1|   Plant A|   Chennai|
|       2|   Plant B| Hyderabad|
|       3|   Plant C|Coimbatore|
+--------+----------+----------+



In [19]:
plant_struct_df.printSchema()

root
 |-- plant_id: integer (nullable = true)
 |-- plant_name: string (nullable = true)
 |-- city: string (nullable = true)



In [20]:
from pyspark.sql import functions as F

In [21]:
plant_struct_df=plant_struct_df.withColumn(
    "country",
    F.lit("India")
)
plant_struct_df.show()


+--------+----------+----------+-------+
|plant_id|plant_name|      city|country|
+--------+----------+----------+-------+
|       1|   Plant A|   Chennai|  India|
|       2|   Plant B| Hyderabad|  India|
|       3|   Plant C|Coimbatore|  India|
+--------+----------+----------+-------+



**3. Pandas DataFrame**

In [22]:
import pandas as pd
pdf = pd.read_csv("manufacturing_production_120k.csv")

In [23]:
pdf.head()

,production_id,production_date,plant_id,machine_id,operator_id,shift,product_code,product_family,units_planned,units_produced,defective_units,downtime_minutes,energy_kwh,temperature_c,vibration_mm_s,quality_status,maintenance_flag,supervisor
0,PR0000001,2026-08-07,PL03,MC011,OP0275,Evening,MTR-100,Motors,1060,993,30,34.0,388.11,61.40,3.88,Rework,No,Charan
1,PR0000002,2026-08-03,PL04,MC019,OP0489,Morning,CTL-330,Controllers,809,836,0,0.0,398.58,64.37,2.28,Reject,No,Farah
2,PR0000003,2026-05-20,PL01,MC002,OP0189,Evening,MTR-300,Motors,442,380,17,15.0,183.34,54.13,4.66,Reject,No,Deepa
3,PR0000004,2026-01-15,PL03,MC013,OP0076,Morning,PMP-210,Pumps,787,720,7,39.0,217.61,64.47,NaN,Rework,No,Farah
4,PR0000005,2026-09-21,PL04,MC020,OP0501,Morning,PMP-110,Pumps,344,296,7,49.0,115.47,72.59,3.21,Pass,No,Deepa


In [24]:
pdf.tail()

,production_id,production_date,plant_id,machine_id,operator_id,shift,product_code,product_family,units_planned,units_produced,defective_units,downtime_minutes,energy_kwh,temperature_c,vibration_mm_s,quality_status,maintenance_flag,supervisor
119995,PR0119996,2026-07-13,PL05,MC022,OP0193,Evening,MTR-200,Motors,435,399,1,73.0,126.07,62.79,5.73,Rework,No,Bharat
119996,PR0119997,2026-04-26,PL05,MC024,OP0124,Evening,CTL-130,Controllers,813,777,8,49.0,318.22,63.05,3.53,Pass,No,Irfan
119997,PR0119998,2026-02-01,PL06,MC027,OP0051,Evening,VLV-120,Valves,650,597,13,0.0,269.46,60.52,3.81,Pass,No,Bharat
119998,PR0119999,2026-01-17,PL01,MC001,OP0452,Evening,MTR-300,Motors,887,899,22,21.0,184.62,64.89,5.20,Pass,Yes,Farah
119999,PR0120000,2026-08-16,PL02,MC007,OP0420,Morning,MTR-200,Motors,547,477,4,43.0,107.95,62.50,3.97,Pass,No,Farah


In [25]:
pdf.shape

(120000, 18)

In [26]:
pdf.columns

Index(['production_id', 'production_date', 'plant_id', 'machine_id',
       'operator_id', 'shift', 'product_code', 'product_family',
       'units_planned', 'units_produced', 'defective_units',
       'downtime_minutes', 'energy_kwh', 'temperature_c', 'vibration_mm_s',
       'quality_status', 'maintenance_flag', 'supervisor'],
      dtype='object')

In [27]:
pdf.dtypes

,0
production_id,object
production_date,object
plant_id,object
machine_id,object
operator_id,object
shift,object
product_code,object
product_family,object
units_planned,int64
units_produced,int64


In [28]:
print(pdf[["plant_id","machine_id","units_produced"]])

       plant_id machine_id  units_produced
0          PL03      MC011             993
1          PL04      MC019             836
2          PL01      MC002             380
3          PL03      MC013             720
4          PL04      MC020             296
...         ...        ...             ...
119995     PL05      MC022             399
119996     PL05      MC024             777
119997     PL06      MC027             597
119998     PL01      MC001             899
119999     PL02      MC007             477

[120000 rows x 3 columns]


In [29]:
renamed_pdf=pdf.rename(columns={"units_produced":"production_output"})
renamed_pdf.head()

,production_id,production_date,plant_id,machine_id,operator_id,shift,product_code,product_family,units_planned,production_output,defective_units,downtime_minutes,energy_kwh,temperature_c,vibration_mm_s,quality_status,maintenance_flag,supervisor
0,PR0000001,2026-08-07,PL03,MC011,OP0275,Evening,MTR-100,Motors,1060,993,30,34.0,388.11,61.40,3.88,Rework,No,Charan
1,PR0000002,2026-08-03,PL04,MC019,OP0489,Morning,CTL-330,Controllers,809,836,0,0.0,398.58,64.37,2.28,Reject,No,Farah
2,PR0000003,2026-05-20,PL01,MC002,OP0189,Evening,MTR-300,Motors,442,380,17,15.0,183.34,54.13,4.66,Reject,No,Deepa
3,PR0000004,2026-01-15,PL03,MC013,OP0076,Morning,PMP-210,Pumps,787,720,7,39.0,217.61,64.47,NaN,Rework,No,Farah
4,PR0000005,2026-09-21,PL04,MC020,OP0501,Morning,PMP-110,Pumps,344,296,7,49.0,115.47,72.59,3.21,Pass,No,Deepa


In [30]:
print(pdf[pdf["quality_status"]=="Reject"])

       production_id production_date plant_id machine_id operator_id    shift  \
1          PR0000002      2026-08-03     PL04      MC019      OP0489  Morning   
2          PR0000003      2026-05-20     PL01      MC002      OP0189  Evening   
7          PR0000008      2026-08-17     PL02      MC009      OP0106    Night   
9          PR0000010      2026-02-13     PL02      MC006      OP0593  Evening   
21         PR0000022      2026-03-03     PL02      MC009      OP0294  Evening   
...              ...             ...      ...        ...         ...      ...   
119979     PR0119980      2026-07-21     PL02      MC006      OP0591  Morning   
119988     PR0119989      2026-01-29     PL01      MC003      OP0021    Night   
119989     PR0119990      2026-07-20     PL06      MC027      OP0217  Evening   
119991     PR0119992      2026-04-07     PL05      MC024      OP0186  Morning   
119994     PR0119995      2026-09-29     PL03      MC013      OP0151  Evening   

       product_code product

In [31]:
print(pdf[pdf["temperature_c"]>75])

       production_id production_date plant_id machine_id operator_id    shift  \
14         PR0000015      2026-03-15     PL02      MC007      OP0238  Morning   
16         PR0000017      2026-08-04     PL03      MC011      OP0533  Morning   
19         PR0000020      2026-02-18     PL03      MC014      OP0563  Morning   
26         PR0000027      2026-01-03     PL05      MC023      OP0190  Morning   
54         PR0000055      2026-07-31     PL01      MC005      OP0212  Morning   
...              ...             ...      ...        ...         ...      ...   
119767     PR0119768      2026-01-29     PL02      MC007      OP0262  Evening   
119810     PR0119811      2026-07-09     PL05      MC024      OP0372  Evening   
119833     PR0119834      2026-06-22     PL04      MC020      OP0038  Morning   
119848     PR0119849      2026-02-25     PL06      MC027      OP0563  Morning   
119968     PR0119969      2026-06-05     PL05      MC022      OP0366    Night   

       product_code product

In [32]:
print(pdf[pdf["shift"]=="Night"])

       production_id production_date plant_id machine_id operator_id  shift  \
7          PR0000008      2026-08-17     PL02      MC009      OP0106  Night   
10         PR0000011      2026-09-29     PL03      MC012      OP0141  Night   
12         PR0000013      2026-07-19     PL06      MC027      OP0172  Night   
28         PR0000029      2026-04-30     PL05      MC023      OP0339  Night   
29         PR0000030      2026-09-18     PL01      MC004      OP0266  Night   
...              ...             ...      ...        ...         ...    ...   
119982     PR0119983      2026-08-10     PL03      MC013      OP0579  Night   
119985     PR0119986      2026-06-22     PL04      MC016      OP0543  Night   
119986     PR0119987      2026-06-10     PL05      MC025      OP0346  Night   
119987     PR0119988      2026-08-18     PL04      MC016      OP0142  Night   
119988     PR0119989      2026-01-29     PL01      MC003      OP0021  Night   

       product_code product_family  units_planned  

**4. Dask DataFrame**

In [33]:
import dask.dataframe as dd
ddf = dd.read_csv(
"manufacturing_production_120k.csv",assume_missing=True
)

In [34]:
print(ddf.head())

  production_id production_date plant_id machine_id operator_id    shift  \
0     PR0000001      2026-08-07     PL03      MC011      OP0275  Evening   
1     PR0000002      2026-08-03     PL04      MC019      OP0489  Morning   
2     PR0000003      2026-05-20     PL01      MC002      OP0189  Evening   
3     PR0000004      2026-01-15     PL03      MC013      OP0076  Morning   
4     PR0000005      2026-09-21     PL04      MC020      OP0501  Morning   

  product_code product_family  units_planned  units_produced  defective_units  \
0      MTR-100         Motors         1060.0           993.0             30.0   
1      CTL-330    Controllers          809.0           836.0              0.0   
2      MTR-300         Motors          442.0           380.0             17.0   
3      PMP-210          Pumps          787.0           720.0              7.0   
4      PMP-110          Pumps          344.0           296.0              7.0   

   downtime_minutes  energy_kwh  temperature_c  vibratio

In [35]:
print(ddf.npartitions)

1


In [36]:
selected=ddf[["plant_id","product_family","units_produced"]]
print(selected.head())

  plant_id product_family  units_produced
0     PL03         Motors           993.0
1     PL04    Controllers           836.0
2     PL01         Motors           380.0
3     PL03          Pumps           720.0
4     PL04          Pumps           296.0


In [37]:
print(ddf[ddf["quality_status"]=="Reject"].head())

   production_id production_date plant_id machine_id operator_id    shift  \
1      PR0000002      2026-08-03     PL04      MC019      OP0489  Morning   
2      PR0000003      2026-05-20     PL01      MC002      OP0189  Evening   
7      PR0000008      2026-08-17     PL02      MC009      OP0106    Night   
9      PR0000010      2026-02-13     PL02      MC006      OP0593  Evening   
21     PR0000022      2026-03-03     PL02      MC009      OP0294  Evening   

   product_code product_family  units_planned  units_produced  \
1       CTL-330    Controllers          809.0           836.0   
2       MTR-300         Motors          442.0           380.0   
7       VLV-320         Valves          862.0           895.0   
9       CTL-230    Controllers          532.0           527.0   
21      MTR-300         Motors          623.0           560.0   

    defective_units  downtime_minutes  energy_kwh  temperature_c  \
1               0.0               0.0      398.58          64.37   
2         

In [38]:
avg_units=ddf.groupby("plant_id")["units_produced"].mean()
print(avg_units.compute())

plant_id
PL01    672.998304
PL02    672.781326
PL03    675.665503
PL04    673.835101
PL05    673.124994
PL06    678.006884
Name: units_produced, dtype: float64


In [39]:
total_defects=ddf.groupby("product_family")["defective_units"].sum()
print(total_defects.compute())

product_family
Controllers    507781.0
Motors         505287.0
Pumps          507864.0
Valves         509312.0
Name: defective_units, dtype: float64


In [40]:
pavg_units=pdf.groupby("plant_id")["units_produced"].mean()
print("Pandas:")
print(pavg_units)
davg_units=ddf.groupby("plant_id")["units_produced"].mean().compute()
print("Dask:")
print(davg_units)

Pandas:
plant_id
PL01    672.998304
PL02    672.781326
PL03    675.665503
PL04    673.835101
PL05    673.124994
PL06    678.006884
Name: units_produced, dtype: float64
Dask:
plant_id
PL01    672.998304
PL02    672.781326
PL03    675.665503
PL04    673.835101
PL05    673.124994
PL06    678.006884
Name: units_produced, dtype: float64


**5. Manipulating DataFrames**

In [41]:
transformed_df=df.withColumn(
    "production_efficiency",
    F.when(F.col("units_planned")!=0,
           F.col("units_produced")/F.col("units_planned")*100)
).withColumn(
    "good_units",
    F.col("units_produced")-F.col("defective_units")
).withColumn(
    "country",
    F.lit("India")
).withColumnRenamed(
    "energy_kwh","energy_consumed"
).withColumn(
    "units_produced",
    F.col("units_produced").cast("double")
).withColumn(
    "defect_percentage",
    F.when(
        F.col("units_produced") != 0,
        F.col("defective_units") / F.col("units_produced") * 100
    )
).withColumn(
    "energy_per_unit",
    F.when(
        F.col("units_produced") != 0,
        F.col("energy_consumed") / F.col("units_produced")
    )
)

transformed_df.select(
    "production_efficiency",
    "good_units",
    "country",
    "energy_consumed",
    "units_produced",
    "defect_percentage",
    "energy_per_unit"
).show()

+---------------------+----------+-------+---------------+--------------+------------------+-------------------+
|production_efficiency|good_units|country|energy_consumed|units_produced| defect_percentage|    energy_per_unit|
+---------------------+----------+-------+---------------+--------------+------------------+-------------------+
|    93.67924528301886|       963|  India|         388.11|         993.0|3.0211480362537766|0.39084592145015107|
|   103.33745364647713|       836|  India|         398.58|         836.0|               0.0| 0.4767703349282296|
|    85.97285067873304|       363|  India|         183.34|         380.0| 4.473684210526316|0.48247368421052633|
|     91.4866581956798|       713|  India|         217.61|         720.0|0.9722222222222222|0.30223611111111115|
|    86.04651162790698|       289|  India|         115.47|         296.0| 2.364864864864865| 0.3901013513513513|
|    82.98538622129436|       745|  India|         290.26|         795.0| 6.289308176100629| 0.3

**6. Dropping Data**

In [42]:
df.drop("supervisor").show()

+-------------+---------------+--------+----------+-----------+-------+------------+--------------+-------------+--------------+---------------+----------------+----------+-------------+--------------+--------------+----------------+
|production_id|production_date|plant_id|machine_id|operator_id|  shift|product_code|product_family|units_planned|units_produced|defective_units|downtime_minutes|energy_kwh|temperature_c|vibration_mm_s|quality_status|maintenance_flag|
+-------------+---------------+--------+----------+-----------+-------+------------+--------------+-------------+--------------+---------------+----------------+----------+-------------+--------------+--------------+----------------+
|    PR0000001|     2026-08-07|    PL03|     MC011|     OP0275|Evening|     MTR-100|        Motors|         1060|           993|             30|              34|    388.11|         61.4|          3.88|        Rework|              No|
|    PR0000002|     2026-08-03|    PL04|     MC019|     OP0489|M

In [43]:
df.drop("supervisor","maintenance_flag").show()

+-------------+---------------+--------+----------+-----------+-------+------------+--------------+-------------+--------------+---------------+----------------+----------+-------------+--------------+--------------+
|production_id|production_date|plant_id|machine_id|operator_id|  shift|product_code|product_family|units_planned|units_produced|defective_units|downtime_minutes|energy_kwh|temperature_c|vibration_mm_s|quality_status|
+-------------+---------------+--------+----------+-----------+-------+------------+--------------+-------------+--------------+---------------+----------------+----------+-------------+--------------+--------------+
|    PR0000001|     2026-08-07|    PL03|     MC011|     OP0275|Evening|     MTR-100|        Motors|         1060|           993|             30|              34|    388.11|         61.4|          3.88|        Rework|
|    PR0000002|     2026-08-03|    PL04|     MC019|     OP0489|Morning|     CTL-330|   Controllers|          809|           836|    

In [44]:
df.dropDuplicates().show()

+-------------+---------------+--------+----------+-----------+-------+------------+--------------+-------------+--------------+---------------+----------------+----------+-------------+--------------+--------------+----------------+----------+
|production_id|production_date|plant_id|machine_id|operator_id|  shift|product_code|product_family|units_planned|units_produced|defective_units|downtime_minutes|energy_kwh|temperature_c|vibration_mm_s|quality_status|maintenance_flag|supervisor|
+-------------+---------------+--------+----------+-----------+-------+------------+--------------+-------------+--------------+---------------+----------------+----------+-------------+--------------+--------------+----------------+----------+
|    PR0000304|     2026-07-31|    PL01|     MC004|     OP0518|Evening|     MTR-200|        Motors|          339|           288|              0|              57|     83.44|        58.53|          1.47|          Pass|              No|    Bharat|
|    PR0000408|     

In [45]:
df.dropDuplicates(["production_id"]).show()

+-------------+---------------+--------+----------+-----------+-------+------------+--------------+-------------+--------------+---------------+----------------+----------+-------------+--------------+--------------+----------------+----------+
|production_id|production_date|plant_id|machine_id|operator_id|  shift|product_code|product_family|units_planned|units_produced|defective_units|downtime_minutes|energy_kwh|temperature_c|vibration_mm_s|quality_status|maintenance_flag|supervisor|
+-------------+---------------+--------+----------+-----------+-------+------------+--------------+-------------+--------------+---------------+----------------+----------+-------------+--------------+--------------+----------------+----------+
|    PR0000002|     2026-08-03|    PL04|     MC019|     OP0489|Morning|     CTL-330|   Controllers|          809|           836|              0|               0|    398.58|        64.37|          2.28|        Reject|              No|     Farah|
|    PR0000003|     

In [46]:
df.filter(
    (F.col("operator_ID").isNotNull()) &
    (F.trim(F.col("operator_ID"))!="")
).show()


+-------------+---------------+--------+----------+-----------+-------+------------+--------------+-------------+--------------+---------------+----------------+----------+-------------+--------------+--------------+----------------+----------+
|production_id|production_date|plant_id|machine_id|operator_id|  shift|product_code|product_family|units_planned|units_produced|defective_units|downtime_minutes|energy_kwh|temperature_c|vibration_mm_s|quality_status|maintenance_flag|supervisor|
+-------------+---------------+--------+----------+-----------+-------+------------+--------------+-------------+--------------+---------------+----------------+----------+-------------+--------------+--------------+----------------+----------+
|    PR0000001|     2026-08-07|    PL03|     MC011|     OP0275|Evening|     MTR-100|        Motors|         1060|           993|             30|              34|    388.11|         61.4|          3.88|        Rework|              No|    Charan|
|    PR0000002|     

In [47]:
df.na.drop(
    how="all",
    subset=["temperature_c","vibration_mm_s"]
).show()

+-------------+---------------+--------+----------+-----------+-------+------------+--------------+-------------+--------------+---------------+----------------+----------+-------------+--------------+--------------+----------------+----------+
|production_id|production_date|plant_id|machine_id|operator_id|  shift|product_code|product_family|units_planned|units_produced|defective_units|downtime_minutes|energy_kwh|temperature_c|vibration_mm_s|quality_status|maintenance_flag|supervisor|
+-------------+---------------+--------+----------+-----------+-------+------------+--------------+-------------+--------------+---------------+----------------+----------+-------------+--------------+--------------+----------------+----------+
|    PR0000001|     2026-08-07|    PL03|     MC011|     OP0275|Evening|     MTR-100|        Motors|         1060|           993|             30|              34|    388.11|         61.4|          3.88|        Rework|              No|    Charan|
|    PR0000002|     

In [48]:
df.select("product_family").distinct().show()

+--------------+
|product_family|
+--------------+
|         Pumps|
|        Valves|
|   Controllers|
|        Motors|
+--------------+



In [49]:
df.select("plant_id","machine_id").distinct().show()

+--------+----------+
|plant_id|machine_id|
+--------+----------+
|    PL01|     MC001|
|    PL01|     MC004|
|    PL04|     MC019|
|    PL04|     MC017|
|    PL05|     MC022|
|    PL04|     MC018|
|    PL01|     MC003|
|    PL06|     MC029|
|    PL02|     MC007|
|    PL03|     MC013|
|    PL06|     MC030|
|    PL04|     MC020|
|    PL05|     MC021|
|    PL01|     MC005|
|    PL02|     MC010|
|    PL04|     MC016|
|    PL06|     MC027|
|    PL02|     MC006|
|    PL01|     MC002|
|    PL06|     MC028|
+--------+----------+
only showing top 20 rows


**7. Sorting**

In [50]:
df.sort(F.col("units_produced").asc()).show()

+-------------+---------------+--------+----------+-----------+-------+------------+--------------+-------------+--------------+---------------+----------------+----------+-------------+--------------+--------------+----------------+----------+
|production_id|production_date|plant_id|machine_id|operator_id|  shift|product_code|product_family|units_planned|units_produced|defective_units|downtime_minutes|energy_kwh|temperature_c|vibration_mm_s|quality_status|maintenance_flag|supervisor|
+-------------+---------------+--------+----------+-----------+-------+------------+--------------+-------------+--------------+---------------+----------------+----------+-------------+--------------+--------------+----------------+----------+
|    PR0038643|     2026-08-06|    PL02|     MC009|     OP0361|Evening|     MTR-100|        Motors|          250|           205|              6|              73|     65.18|        55.99|          2.72|          Pass|              No|    Harini|
|    PR0059668|     

In [51]:
df.sort(F.col("units_produced").desc()).show()

+-------------+---------------+--------+----------+-----------+-------+------------+--------------+-------------+--------------+---------------+----------------+----------+-------------+--------------+--------------+----------------+----------+
|production_id|production_date|plant_id|machine_id|operator_id|  shift|product_code|product_family|units_planned|units_produced|defective_units|downtime_minutes|energy_kwh|temperature_c|vibration_mm_s|quality_status|maintenance_flag|supervisor|
+-------------+---------------+--------+----------+-----------+-------+------------+--------------+-------------+--------------+---------------+----------------+----------+-------------+--------------+--------------+----------------+----------+
|    PR0087560|     2026-06-29|    PL04|     MC018|     OP0587|  Night|     VLV-120|        Valves|         1199|          1245|             60|               0|    500.87|        58.11|          3.58|          Pass|             Yes|    Charan|
|    PR0008535|     

In [52]:
df.sort(F.col("defective_units").desc()).show()

+-------------+---------------+--------+----------+-----------+-------+------------+--------------+-------------+--------------+---------------+----------------+----------+-------------+--------------+--------------+----------------+----------+
|production_id|production_date|plant_id|machine_id|operator_id|  shift|product_code|product_family|units_planned|units_produced|defective_units|downtime_minutes|energy_kwh|temperature_c|vibration_mm_s|quality_status|maintenance_flag|supervisor|
+-------------+---------------+--------+----------+-----------+-------+------------+--------------+-------------+--------------+---------------+----------------+----------+-------------+--------------+--------------+----------------+----------+
|    PR0051359|     2026-09-16|    PL02|     MC009|     OP0181|Morning|     CTL-230|   Controllers|         1192|          1230|            108|               8|    555.34|        71.97|          2.09|        Rework|              No|     Irfan|
|    PR0108009|     

In [53]:
df.sort("plant_id",F.col("units_produced").desc()).show()

+-------------+---------------+--------+----------+-----------+-------+------------+--------------+-------------+--------------+---------------+----------------+----------+-------------+--------------+--------------+----------------+----------+
|production_id|production_date|plant_id|machine_id|operator_id|  shift|product_code|product_family|units_planned|units_produced|defective_units|downtime_minutes|energy_kwh|temperature_c|vibration_mm_s|quality_status|maintenance_flag|supervisor|
+-------------+---------------+--------+----------+-----------+-------+------------+--------------+-------------+--------------+---------------+----------------+----------+-------------+--------------+--------------+----------------+----------+
|    PR0073059|     2026-04-01|    PL01|     MC003|     OP0110|Evening|     MTR-200|        Motors|         1198|          1239|             11|               0|     282.6|        50.32|          3.91|        Rework|              No|     Gopal|
|    PR0002678|     

In [54]:
df.sort(F.col("units_produced").desc()).show(20)

+-------------+---------------+--------+----------+-----------+-------+------------+--------------+-------------+--------------+---------------+----------------+----------+-------------+--------------+--------------+----------------+----------+
|production_id|production_date|plant_id|machine_id|operator_id|  shift|product_code|product_family|units_planned|units_produced|defective_units|downtime_minutes|energy_kwh|temperature_c|vibration_mm_s|quality_status|maintenance_flag|supervisor|
+-------------+---------------+--------+----------+-----------+-------+------------+--------------+-------------+--------------+---------------+----------------+----------+-------------+--------------+--------------+----------------+----------+
|    PR0087560|     2026-06-29|    PL04|     MC018|     OP0587|  Night|     VLV-120|        Valves|         1199|          1245|             60|               0|    500.87|        58.11|          3.58|          Pass|             Yes|    Charan|
|    PR0008535|     

In [55]:
df.sort(F.col("downtime_minutes").desc()).show(10)

+-------------+---------------+--------+----------+-----------+-------+------------+--------------+-------------+--------------+---------------+----------------+----------+-------------+--------------+--------------+----------------+----------+
|production_id|production_date|plant_id|machine_id|operator_id|  shift|product_code|product_family|units_planned|units_produced|defective_units|downtime_minutes|energy_kwh|temperature_c|vibration_mm_s|quality_status|maintenance_flag|supervisor|
+-------------+---------------+--------+----------+-----------+-------+------------+--------------+-------------+--------------+---------------+----------------+----------+-------------+--------------+--------------+----------------+----------+
|    PR0092818|     2026-02-16|    PL06|     MC027|     OP0016|Morning|     PMP-210|         Pumps|         1066|           907|              2|             281|    359.48|        54.03|           2.8|          Pass|              No|     Farah|
|    PR0045363|     

In [56]:
df.sort(F.col("vibration_mm_s").desc()).select("machine_id","vibration_mm_s").show()

+----------+--------------+
|machine_id|vibration_mm_s|
+----------+--------------+
|     MC012|          8.56|
|     MC030|          8.25|
|     MC027|          7.85|
|     MC010|           7.8|
|     MC018|          7.79|
|     MC024|          7.77|
|     MC015|          7.76|
|     MC027|          7.75|
|     MC015|          7.67|
|     MC016|          7.63|
|     MC007|          7.61|
|     MC010|          7.59|
|     MC027|          7.58|
|     MC017|          7.52|
|     MC021|           7.5|
|     MC021|           7.5|
|     MC004|          7.48|
|     MC027|          7.47|
|     MC025|          7.46|
|     MC019|          7.45|
+----------+--------------+
only showing top 20 rows


**8. Aggregations**

In [58]:
df.agg(
    F.sum("units_planned").alias("total_planned"),
    F.sum("units_produced").alias("total_produced"),
    F.sum("defective_units").alias("total_defects"),
    F.avg("units_produced").alias("average_production"),
    F.max("units_produced").alias("maximum_production"),
    F.min("units_produced").alias("minimum_production"),
    F.avg("downtime_minutes").alias("average_downtime"),
    F.sum("energy_kwh").alias("total_energy"),
    F.avg("temperature_c").alias("average_temperature"),
    F.avg("vibration_mm_s").alias("average_vibration")
).show()

+-------------+--------------+-------------+------------------+------------------+------------------+-----------------+--------------------+-------------------+-----------------+
|total_planned|total_produced|total_defects|average_production|maximum_production|minimum_production| average_downtime|        total_energy|average_temperature|average_vibration|
+-------------+--------------+-------------+------------------+------------------+------------------+-----------------+--------------------+-------------------+-----------------+
|     87094621|      80927888|      2030244| 674.3990666666666|              1245|               205|33.93149474428351|2.8368014080000117E7| 63.021532392167266|3.398737691604917|
+-------------+--------------+-------------+------------------+------------------+------------------+-----------------+--------------------+-------------------+-----------------+



9.groupBy()

In [60]:
from pyspark.sql.functions import col

In [62]:
df.groupby("plant_id").agg(
    F.sum(col("units_produced")).alias("total_production_by_plant"),
).orderBy(col("total_production_by_plant").desc()).show()

+--------+-------------------------+
|plant_id|total_production_by_plant|
+--------+-------------------------+
|    PL03|                 13547769|
|    PL02|                 13546452|
|    PL04|                 13529935|
|    PL06|                 13492337|
|    PL01|                 13488232|
|    PL05|                 13323163|
+--------+-------------------------+



In [63]:
df.groupby("product_family").agg(
    F.sum(col("units_produced")).alias("total_production_by_product_family"),
).orderBy(col("total_production_by_product_family").desc()).show()

+--------------+----------------------------------+
|product_family|total_production_by_product_family|
+--------------+----------------------------------+
|        Valves|                          20433082|
|         Pumps|                          20208739|
|   Controllers|                          20168572|
|        Motors|                          20117495|
+--------------+----------------------------------+



In [64]:
df.groupby("plant_id").agg(
    F.sum(col("defective_units")).alias("total_defects_by_plant"),
).orderBy(col("total_defects_by_plant").desc()).show()

+--------+----------------------+
|plant_id|total_defects_by_plant|
+--------+----------------------+
|    PL01|                339720|
|    PL02|                339647|
|    PL03|                339264|
|    PL06|                338993|
|    PL04|                338118|
|    PL05|                334502|
+--------+----------------------+



In [66]:
df.groupby("shift").agg(
    F.avg(col("downtime_minutes")).alias("avg_downtime_by_shift"),
).orderBy(col("avg_downtime_by_shift").desc()).show()

+-------+---------------------+
|  shift|avg_downtime_by_shift|
+-------+---------------------+
|Evening|     34.2213017151833|
|  Night|   33.953058820580935|
|Morning|    33.61973080530278|
+-------+---------------------+



In [68]:
df.groupby("product_family").agg(
    F.avg(col("energy_kwh")).alias("avg_energy_usage_by_product_family"),
).orderBy(col("avg_energy_usage_by_product_family").desc()).show()

+--------------+----------------------------------+
|product_family|avg_energy_usage_by_product_family|
+--------------+----------------------------------+
|        Motors|                237.21818224597516|
|         Pumps|                236.89137927582252|
|        Valves|                236.34508750495314|
|   Controllers|                235.15370433333288|
+--------------+----------------------------------+



In [71]:
df.groupby("quality_status").agg(
    F.count(col("production_id")).alias("production_count_by_quality_status"),
).orderBy(col("production_count_by_quality_status").desc()).show()

+--------------+----------------------------------+
|quality_status|production_count_by_quality_status|
+--------------+----------------------------------+
|          Pass|                             74328|
|        Rework|                             22928|
|        Reject|                             22744|
+--------------+----------------------------------+



In [72]:
df.groupby("plant_id","shift").agg(
    F.count(col("production_id")).alias("production_count_by_plant_and_shift"),
).orderBy(col("production_count_by_plant_and_shift").desc()).show()

+--------+-------+-----------------------------------+
|plant_id|  shift|production_count_by_plant_and_shift|
+--------+-------+-----------------------------------+
|    PL03|  Night|                               6790|
|    PL02|  Night|                               6788|
|    PL02|Morning|                               6768|
|    PL06|  Night|                               6758|
|    PL01|  Night|                               6745|
|    PL04|Evening|                               6715|
|    PL04|Morning|                               6703|
|    PL03|Evening|                               6672|
|    PL01|Evening|                               6668|
|    PL04|  Night|                               6661|
|    PL01|Morning|                               6629|
|    PL05|Evening|                               6628|
|    PL06|Evening|                               6601|
|    PL05|Morning|                               6600|
|    PL03|Morning|                               6589|
|    PL02|

In [74]:
df.groupby("plant_id","product_family").agg(
    F.count(col("defective_units")).alias("defects_by_plant_and_product_family"),
).orderBy(col("defects_by_plant_and_product_family").desc()).show()

+--------+--------------+-----------------------------------+
|plant_id|product_family|defects_by_plant_and_product_family|
+--------+--------------+-----------------------------------+
|    PL01|        Valves|                               5128|
|    PL04|        Valves|                               5125|
|    PL03|        Valves|                               5075|
|    PL02|        Valves|                               5057|
|    PL06|   Controllers|                               5055|
|    PL01|         Pumps|                               5055|
|    PL02|   Controllers|                               5050|
|    PL04|        Motors|                               5026|
|    PL03|         Pumps|                               5020|
|    PL02|         Pumps|                               5016|
|    PL06|        Valves|                               5014|
|    PL02|        Motors|                               5012|
|    PL05|        Motors|                               5005|
|    PL0

In [75]:
df.groupby("machine_id").agg(
    F.avg(col("temperature_c")).alias("avg_temperature_by_machine"),
).orderBy(col("avg_temperature_by_machine").desc()).show()


+----------+--------------------------+
|machine_id|avg_temperature_by_machine|
+----------+--------------------------+
|     MC019|        63.266223444138944|
|     MC020|         63.22489208633087|
|     MC029|         63.13119731057668|
|     MC009|          63.1252779855396|
|     MC014|        63.114309957574186|
|     MC023|        63.109051746112684|
|     MC027|        63.099668282725126|
|     MC022|         63.09403431245128|
|     MC015|         63.09026488395557|
|     MC001|          63.0778733146782|
|     MC017|         63.07674442190667|
|     MC011|        63.060230414746606|
|     MC007|           63.037661716587|
|     MC016|         63.03639870582378|
|     MC026|         63.02672521957329|
|     MC021|         63.01464069491981|
|     MC005|        62.979236446933044|
|     MC028|         62.97779683034558|
|     MC030|         62.96733774497327|
|     MC024|         62.96457946271644|
+----------+--------------------------+
only showing top 20 rows


In [76]:
df.orderBy("supervisor").agg(
    F.count(col("production_id")).alias("production_count_by_supervisor"),
).orderBy(col("production_count_by_supervisor").desc()).show()

+------------------------------+
|production_count_by_supervisor|
+------------------------------+
|                        120000|
+------------------------------+



**10. Joining DataFrames**

In [77]:
plant_df = spark.read.csv(
"plant_master.csv",
    header=True,
    inferSchema=True
)
machine_df = spark.read.csv(
"machine_master.csv",
    header=True,
    inferSchema=True
)

In [78]:
joined_plant = df.join(
    plant_df,
    on="plant_id",
    how="inner"
)

joined_plant.show()


+--------+-------------+---------------+----------+-----------+-------+------------+--------------+-------------+--------------+---------------+----------------+----------+-------------+--------------+--------------+----------------+----------+---------------+---------+------+-----------+-------------+
|plant_id|production_id|production_date|machine_id|operator_id|  shift|product_code|product_family|units_planned|units_produced|defective_units|downtime_minutes|energy_kwh|temperature_c|vibration_mm_s|quality_status|maintenance_flag|supervisor|     plant_name|     city|region|year_opened|plant_manager|
+--------+-------------+---------------+----------+-----------+-------+------------+--------------+-------------+--------------+---------------+----------------+----------+-------------+--------------+--------------+----------------+----------+---------------+---------+------+-----------+-------------+
|    PL03|    PR0000001|     2026-08-07|     MC011|     OP0275|Evening|     MTR-100|    

In [79]:

joined_plant.select(
    "production_id",
    "plant_name",
    "units_produced"
).show()


+-------------+---------------+--------------+
|production_id|     plant_name|units_produced|
+-------------+---------------+--------------+
|    PR0000001|  Chennai Plant|           993|
|    PR0000002|    Noida Plant|           836|
|    PR0000003|Hyderabad Plant|           380|
|    PR0000004|  Chennai Plant|           720|
|    PR0000005|    Noida Plant|           296|
|    PR0000006|Hyderabad Plant|           795|
|    PR0000007|     Pune Plant|           955|
|    PR0000008|     Pune Plant|           895|
|    PR0000009|  Chennai Plant|           289|
|    PR0000010|     Pune Plant|           527|
|    PR0000011|  Chennai Plant|           512|
|    PR0000012|  Kolkata Plant|           968|
|    PR0000013|  Kolkata Plant|           867|
|    PR0000014|  Chennai Plant|           456|
|    PR0000015|     Pune Plant|          1042|
|    PR0000016|    Noida Plant|           732|
|    PR0000017|  Chennai Plant|           963|
|    PR0000018|  Chennai Plant|           469|
|    PR000001

In [80]:

joined_machine = df.join(
    machine_df,
    on="machine_id",
    how="inner"
)

joined_machine.show()


+----------+-------------+---------------+--------+-----------+-------+------------+--------------+-------------+--------------+---------------+----------------+----------+-------------+--------------+--------------+----------------+----------+--------+------------+-----------------+--------------+
|machine_id|production_id|production_date|plant_id|operator_id|  shift|product_code|product_family|units_planned|units_produced|defective_units|downtime_minutes|energy_kwh|temperature_c|vibration_mm_s|quality_status|maintenance_flag|supervisor|plant_id|machine_type|capacity_per_hour|installed_year|
+----------+-------------+---------------+--------+-----------+-------+------------+--------------+-------------+--------------+---------------+----------------+----------+-------------+--------------+--------------+----------------+----------+--------+------------+-----------------+--------------+
|     MC011|    PR0000001|     2026-08-07|    PL03|     OP0275|Evening|     MTR-100|        Motors| 

In [82]:
full_df = df.join(
    plant_df, "plant_id", "left"
).join(
    machine_df, "machine_id", "left"
)



In [83]:

full_df.groupBy("city").agg(
    F.sum("units_produced").alias("total")
).show()


+---------+--------+
|     city|   total|
+---------+--------+
|  Chennai|13547769|
|Ahmedabad|13323163|
|  Kolkata|13492337|
|    Noida|13529935|
|     Pune|13546452|
|Hyderabad|13488232|
+---------+--------+



In [84]:
full_df.groupBy("machine_type").agg(
    F.avg("downtime_minutes").alias("avg_downtime")
).show()

+------------+-----------------+
|machine_type|     avg_downtime|
+------------+-----------------+
|         CNC|33.64548129466162|
|     Testing|33.87585594724829|
|     Welding|34.00449382099613|
|    Assembly|33.96718284443325|
|     Casting| 34.1660783396419|
+------------+-----------------+



In [86]:
full_df.groupBy("region").agg(
    F.sum("defective_units").alias("defects")
).show()

+------+-------+
|region|defects|
+------+-------+
| South| 678984|
|  East| 338993|
|  West| 674149|
| North| 338118|
+------+-------+



In [87]:
df.join(
    machine_df,
    on="machine_id",
    how="left_anti"
).show()

+----------+-------------+---------------+--------+-----------+-----+------------+--------------+-------------+--------------+---------------+----------------+----------+-------------+--------------+--------------+----------------+----------+
|machine_id|production_id|production_date|plant_id|operator_id|shift|product_code|product_family|units_planned|units_produced|defective_units|downtime_minutes|energy_kwh|temperature_c|vibration_mm_s|quality_status|maintenance_flag|supervisor|
+----------+-------------+---------------+--------+-----------+-----+------------+--------------+-------------+--------------+---------------+----------------+----------+-------------+--------------+--------------+----------------+----------+
+----------+-------------+---------------+--------+-----------+-----+------------+--------------+-------------+--------------+---------------+----------------+----------+-------------+--------------+--------------+----------------+----------+



In [88]:
left_df = df.join(
    machine_df,
    on="machine_id",
    how="left"
)

left_df.filter(
    F.col("machine_type").isNull()
).show()

+----------+-------------+---------------+--------+-----------+-----+------------+--------------+-------------+--------------+---------------+----------------+----------+-------------+--------------+--------------+----------------+----------+--------+------------+-----------------+--------------+
|machine_id|production_id|production_date|plant_id|operator_id|shift|product_code|product_family|units_planned|units_produced|defective_units|downtime_minutes|energy_kwh|temperature_c|vibration_mm_s|quality_status|maintenance_flag|supervisor|plant_id|machine_type|capacity_per_hour|installed_year|
+----------+-------------+---------------+--------+-----------+-----+------------+--------------+-------------+--------------+---------------+----------------+----------+-------------+--------------+--------------+----------------+----------+--------+------------+-----------------+--------------+
+----------+-------------+---------------+--------+-----------+-----+------------+--------------+---------

In [89]:
full_df.groupBy("plant_name").agg(
    F.sum("energy_kwh").alias("total_energy")
).show()

+---------------+-----------------+
|     plant_name|     total_energy|
+---------------+-----------------+
|    Noida Plant|4746238.580000002|
|  Kolkata Plant|4723180.160000009|
|  Chennai Plant|4742748.169999995|
|     Pune Plant|4757984.949999988|
|Hyderabad Plant|4725471.189999996|
|Ahmedabad Plant| 4672391.03000001|
+---------------+-----------------+



**11. Updating Data**

In [90]:
updated_df = df

In [91]:
updated_df = updated_df.withColumn(
    "quality_status",
    F.when(
        F.col("quality_status") == "Reject", "Failed"
    ).when(
        F.col("quality_status") == "Rework",
        "Requires Review"
    ).otherwise(F.col("quality_status"))
)

In [92]:
updated_df = updated_df.withColumn(
    "units_planned",
    F.when(
        F.col("product_family") == "Motors",
        F.col("units_planned") * 1.05
    ).otherwise(F.col("units_planned"))
)


In [93]:
updated_df = updated_df.withColumn(
    "energy_kwh",
    F.when(
        F.col("shift") == "Night",
        F.col("energy_kwh") * 1.03
    ).otherwise(F.col("energy_kwh"))
)

In [94]:
updated_df = updated_df.withColumn(
    "operator_id",
    F.when(
        F.col("operator_id").isNull() |
        (F.trim(F.col("operator_id")) == ""),
        "UNASSIGNED"
    ).otherwise(F.col("operator_id"))
)

In [95]:
updated_df = updated_df.withColumn(
    "supervisor",
    F.when(
        F.col("supervisor") == "Anita",
        "Anita Rao"
    ).otherwise(F.col("supervisor"))
)

In [97]:
updated_df = updated_df.withColumn(
    "corrected_temperature",
    F.round("temperature_c", 1)
)

updated_df.show()

+-------------+---------------+--------+----------+-----------+-------+------------+--------------+------------------+--------------+---------------+----------------+----------+-------------+--------------+---------------+----------------+----------+---------------------+
|production_id|production_date|plant_id|machine_id|operator_id|  shift|product_code|product_family|     units_planned|units_produced|defective_units|downtime_minutes|energy_kwh|temperature_c|vibration_mm_s| quality_status|maintenance_flag|supervisor|corrected_temperature|
+-------------+---------------+--------+----------+-----------+-------+------------+--------------+------------------+--------------+---------------+----------------+----------+-------------+--------------+---------------+----------------+----------+---------------------+
|    PR0000001|     2026-08-07|    PL03|     MC011|     OP0275|Evening|     MTR-100|        Motors|            1113.0|           993|             30|              34|    388.11|    

12. Null Values and na Functions

In [98]:
df.filter(
    F.col("operator_id").isNull() |
    (F.trim(F.col("operator_id")) == "")
).count()

1524

In [100]:
df.filter(F.col("temperature_c").isNull()).count()

2132

In [101]:
df.filter(F.col("vibration_mm_s").isNull()).count()

1788

In [102]:
df.filter(F.col("downtime_minutes").isNull()).count()

1177

In [103]:
df.na.drop().show()

+-------------+---------------+--------+----------+-----------+-------+------------+--------------+-------------+--------------+---------------+----------------+----------+-------------+--------------+--------------+----------------+----------+
|production_id|production_date|plant_id|machine_id|operator_id|  shift|product_code|product_family|units_planned|units_produced|defective_units|downtime_minutes|energy_kwh|temperature_c|vibration_mm_s|quality_status|maintenance_flag|supervisor|
+-------------+---------------+--------+----------+-----------+-------+------------+--------------+-------------+--------------+---------------+----------------+----------+-------------+--------------+--------------+----------------+----------+
|    PR0000001|     2026-08-07|    PL03|     MC011|     OP0275|Evening|     MTR-100|        Motors|         1060|           993|             30|              34|    388.11|         61.4|          3.88|        Rework|              No|    Charan|
|    PR0000002|     

In [105]:
df.na.drop(subset=["temperature_c"]).show()

+-------------+---------------+--------+----------+-----------+-------+------------+--------------+-------------+--------------+---------------+----------------+----------+-------------+--------------+--------------+----------------+----------+
|production_id|production_date|plant_id|machine_id|operator_id|  shift|product_code|product_family|units_planned|units_produced|defective_units|downtime_minutes|energy_kwh|temperature_c|vibration_mm_s|quality_status|maintenance_flag|supervisor|
+-------------+---------------+--------+----------+-----------+-------+------------+--------------+-------------+--------------+---------------+----------------+----------+-------------+--------------+--------------+----------------+----------+
|    PR0000001|     2026-08-07|    PL03|     MC011|     OP0275|Evening|     MTR-100|        Motors|         1060|           993|             30|              34|    388.11|         61.4|          3.88|        Rework|              No|    Charan|
|    PR0000002|     

In [106]:
df.na.fill({"downtime_minutes": 0}).show()

+-------------+---------------+--------+----------+-----------+-------+------------+--------------+-------------+--------------+---------------+----------------+----------+-------------+--------------+--------------+----------------+----------+
|production_id|production_date|plant_id|machine_id|operator_id|  shift|product_code|product_family|units_planned|units_produced|defective_units|downtime_minutes|energy_kwh|temperature_c|vibration_mm_s|quality_status|maintenance_flag|supervisor|
+-------------+---------------+--------+----------+-----------+-------+------------+--------------+-------------+--------------+---------------+----------------+----------+-------------+--------------+--------------+----------------+----------+
|    PR0000001|     2026-08-07|    PL03|     MC011|     OP0275|Evening|     MTR-100|        Motors|         1060|           993|             30|              34|    388.11|         61.4|          3.88|        Rework|              No|    Charan|
|    PR0000002|     

In [107]:
clean_df = df.withColumn(
    "operator_id",
    F.when(
        F.col("operator_id").isNull() |
        (F.trim(F.col("operator_id")) == ""),
        "UNKNOWN"
    ).otherwise(F.col("operator_id"))
)

In [109]:
avg_temp = df.select(
    F.avg("temperature_c")
).first()[0]

clean_df = clean_df.na.fill({
    "temperature_c": avg_temp
})

In [110]:
print("Before:", df.count())
print("After:", clean_df.count())

Before: 120000
After: 120000


13. Built-in Functions

In [111]:
df.select(
    F.upper("product_family").alias("upper_family")
).show()

+------------+
|upper_family|
+------------+
|      MOTORS|
| CONTROLLERS|
|      MOTORS|
|       PUMPS|
|       PUMPS|
|      MOTORS|
|      VALVES|
|      VALVES|
| CONTROLLERS|
| CONTROLLERS|
|       PUMPS|
| CONTROLLERS|
| CONTROLLERS|
|      VALVES|
|      MOTORS|
| CONTROLLERS|
| CONTROLLERS|
|      MOTORS|
|      VALVES|
|       PUMPS|
+------------+
only showing top 20 rows


In [112]:
df.select(
    F.lower("supervisor").alias("lower_supervisor")
).show()


+----------------+
|lower_supervisor|
+----------------+
|          charan|
|           farah|
|           deepa|
|           farah|
|           deepa|
|           gopal|
|           gopal|
|           irfan|
|           anita|
|           gopal|
|          bharat|
|           gopal|
|           deepa|
|           deepa|
|           deepa|
|           irfan|
|           gopal|
|           farah|
|           deepa|
|           irfan|
+----------------+
only showing top 20 rows


In [113]:
df.select(
    F.trim("operator_id").alias("clean_operator")
).show()

+--------------+
|clean_operator|
+--------------+
|        OP0275|
|        OP0489|
|        OP0189|
|        OP0076|
|        OP0501|
|        OP0501|
|        OP0645|
|        OP0106|
|        OP0386|
|        OP0593|
|        OP0141|
|        OP0047|
|        OP0172|
|        OP0446|
|        OP0238|
|        OP0013|
|        OP0533|
|        OP0162|
|        OP0526|
|        OP0563|
+--------------+
only showing top 20 rows


In [114]:
df.select(
    F.length("product_code").alias("code_length")
).show()

+-----------+
|code_length|
+-----------+
|          7|
|          7|
|          7|
|          7|
|          7|
|          7|
|          7|
|          7|
|          7|
|          7|
|          7|
|          7|
|          7|
|          7|
|          7|
|          7|
|          7|
|          7|
|          7|
|          7|
+-----------+
only showing top 20 rows


In [115]:
df.select(
    F.concat_ws(
        "-",
        F.col("plant_id"),
        F.col("machine_id")
    ).alias("plant_machine")
).show()

+-------------+
|plant_machine|
+-------------+
|   PL03-MC011|
|   PL04-MC019|
|   PL01-MC002|
|   PL03-MC013|
|   PL04-MC020|
|   PL01-MC002|
|   PL02-MC007|
|   PL02-MC009|
|   PL03-MC015|
|   PL02-MC006|
|   PL03-MC012|
|   PL06-MC030|
|   PL06-MC027|
|   PL03-MC013|
|   PL02-MC007|
|   PL04-MC020|
|   PL03-MC011|
|   PL03-MC012|
|   PL01-MC005|
|   PL03-MC014|
+-------------+
only showing top 20 rows


In [117]:
df.select(
    F.round("temperature_c", 1).alias("rounded_temp")
).show()

+------------+
|rounded_temp|
+------------+
|        61.4|
|        64.4|
|        54.1|
|        64.5|
|        72.6|
|        64.9|
|        62.8|
|        57.1|
|        66.6|
|        61.0|
|        52.8|
|        69.8|
|        50.9|
|        67.2|
|        76.4|
|        44.9|
|        75.1|
|        59.9|
|        62.2|
|        76.9|
+------------+
only showing top 20 rows


In [118]:
df.select(
    F.abs(
        F.col("units_planned") -
        F.col("units_produced")
    ).alias("difference")
).show()

+----------+
|difference|
+----------+
|        67|
|        27|
|        62|
|        67|
|        48|
|       163|
|       210|
|        33|
|        24|
|         5|
|        65|
|        12|
|       137|
|         8|
|        44|
|       138|
|       189|
|        72|
|         1|
|        73|
+----------+
only showing top 20 rows


In [119]:
df.select(
    F.regexp_replace(
        "production_id", "-", "_"
    ).alias("modified_id")
).show()

+-----------+
|modified_id|
+-----------+
|  PR0000001|
|  PR0000002|
|  PR0000003|
|  PR0000004|
|  PR0000005|
|  PR0000006|
|  PR0000007|
|  PR0000008|
|  PR0000009|
|  PR0000010|
|  PR0000011|
|  PR0000012|
|  PR0000013|
|  PR0000014|
|  PR0000015|
|  PR0000016|
|  PR0000017|
|  PR0000018|
|  PR0000019|
|  PR0000020|
+-----------+
only showing top 20 rows


In [120]:
df.select(
    F.substring("product_code", 1, 3).alias("code_part")
).show()

+---------+
|code_part|
+---------+
|      MTR|
|      CTL|
|      MTR|
|      PMP|
|      PMP|
|      MTR|
|      VLV|
|      VLV|
|      CTL|
|      CTL|
|      PMP|
|      CTL|
|      CTL|
|      VLV|
|      MTR|
|      CTL|
|      CTL|
|      MTR|
|      VLV|
|      PMP|
+---------+
only showing top 20 rows


In [122]:
df.select(
    F.coalesce(
        F.col("temperature_c"),
        F.lit(0)
    ).alias("temperature_value")
).show()

+-----------------+
|temperature_value|
+-----------------+
|             61.4|
|            64.37|
|            54.13|
|            64.47|
|            72.59|
|            64.91|
|            62.76|
|             57.1|
|            66.55|
|            60.96|
|            52.81|
|            69.79|
|            50.88|
|             67.2|
|            76.39|
|            44.92|
|            75.09|
|            59.91|
|            62.23|
|            76.92|
+-----------------+
only showing top 20 rows


14.
when() / otherwise()

In [123]:
category_df = df.withColumn(
    "efficiency",
    F.when(
        F.col("units_planned") != 0,
        F.col("units_produced") / F.col("units_planned") * 100
    )
)

category_df = category_df.withColumn(
    "efficiency_category",
    F.when(F.col("efficiency") >= 100, "Above Target")
     .when(F.col("efficiency") >= 95, "On Target")
     .when(F.col("efficiency") >= 85, "Below Target")
     .otherwise("Critical")
)

In [124]:
category_df = category_df.withColumn(
    "defect_category",
    F.when(F.col("defective_units") == 0, "Perfect")
     .when(F.col("defective_units") <= 10, "Normal")
     .when(F.col("defective_units") <= 25, "Attention")
     .otherwise("High Defect")
)

In [126]:
category_df = category_df.withColumn(
    "machine_condition",
    F.when(
        (F.col("temperature_c") > 75) |
        (F.col("vibration_mm_s") > 5),
        "Inspect"
    ).otherwise("Normal")
)

category_df.show()

+-------------+---------------+--------+----------+-----------+-------+------------+--------------+-------------+--------------+---------------+----------------+----------+-------------+--------------+--------------+----------------+----------+------------------+-------------------+---------------+-----------------+
|production_id|production_date|plant_id|machine_id|operator_id|  shift|product_code|product_family|units_planned|units_produced|defective_units|downtime_minutes|energy_kwh|temperature_c|vibration_mm_s|quality_status|maintenance_flag|supervisor|        efficiency|efficiency_category|defect_category|machine_condition|
+-------------+---------------+--------+----------+-----------+-------+------------+--------------+-------------+--------------+---------------+----------------+----------+-------------+--------------+--------------+----------------+----------+------------------+-------------------+---------------+-----------------+
|    PR0000001|     2026-08-07|    PL03|     M

In [127]:
category_df.groupBy("efficiency_category").count().show()

category_df.groupBy("defect_category").count().show()

category_df.groupBy("machine_condition").count().show()

+-------------------+-----+
|efficiency_category|count|
+-------------------+-----+
|       Above Target|21735|
|          On Target|26825|
|       Below Target|54619|
|           Critical|16821|
+-------------------+-----+

+---------------+-----+
|defect_category|count|
+---------------+-----+
|      Attention|45745|
|        Perfect|11491|
|    High Defect|28058|
|         Normal|34706|
+---------------+-----+

+-----------------+------+
|machine_condition| count|
+-----------------+------+
|          Inspect| 15578|
|           Normal|104422|
+-----------------+------+



15. Complex JSON

In [129]:
machine_json = spark.read.option(
"multiline", "true"
).json("machine_configurations.json")

In [130]:
machine_json.printSchema()

root
 |-- capacity_per_hour: long (nullable = true)
 |-- installed_year: long (nullable = true)
 |-- machine_id: string (nullable = true)
 |-- machine_type: string (nullable = true)
 |-- maintenance: struct (nullable = true)
 |    |-- frequency_days: long (nullable = true)
 |    |-- last_service: string (nullable = true)
 |-- plant_id: string (nullable = true)
 |-- sensors: array (nullable = true)
 |    |-- element: struct (containsNull = true)
 |    |    |-- active: boolean (nullable = true)
 |    |    |-- sensor: string (nullable = true)
 |    |    |-- unit: string (nullable = true)
 |-- specifications: struct (nullable = true)
 |    |-- max_temperature_c: long (nullable = true)
 |    |-- rated_power_kw: long (nullable = true)
 |    |-- vibration_limit_mm_s: double (nullable = true)



In [131]:
machine_json.select(
    "machine_id",
    "machine_type"
).show()

+----------+------------+
|machine_id|machine_type|
+----------+------------+
|     MC001|         CNC|
|     MC002|    Assembly|
|     MC003|     Casting|
|     MC004|     Welding|
|     MC005|     Testing|
|     MC006|         CNC|
|     MC007|    Assembly|
|     MC008|     Casting|
|     MC009|     Welding|
|     MC010|     Testing|
|     MC011|         CNC|
|     MC012|    Assembly|
|     MC013|     Casting|
|     MC014|     Welding|
|     MC015|     Testing|
|     MC016|         CNC|
|     MC017|    Assembly|
|     MC018|     Casting|
|     MC019|     Welding|
|     MC020|     Testing|
+----------+------------+
only showing top 20 rows


In [132]:
machine_json.select(
    "machine_id",
    "specifications.rated_power_kw"
).show()

+----------+--------------+
|machine_id|rated_power_kw|
+----------+--------------+
|     MC001|            22|
|     MC002|            22|
|     MC003|            55|
|     MC004|            22|
|     MC005|            30|
|     MC006|            30|
|     MC007|            22|
|     MC008|            55|
|     MC009|            30|
|     MC010|            55|
|     MC011|            30|
|     MC012|            55|
|     MC013|            30|
|     MC014|            30|
|     MC015|            45|
|     MC016|            30|
|     MC017|            30|
|     MC018|            30|
|     MC019|            30|
|     MC020|            15|
+----------+--------------+
only showing top 20 rows


In [134]:
machine_json.select(
    "machine_id",
    "specifications.max_temperature_c"
).show()


+----------+-----------------+
|machine_id|max_temperature_c|
+----------+-----------------+
|     MC001|               85|
|     MC002|               85|
|     MC003|               90|
|     MC004|               85|
|     MC005|               85|
|     MC006|               85|
|     MC007|               80|
|     MC008|               75|
|     MC009|               85|
|     MC010|               90|
|     MC011|               90|
|     MC012|               80|
|     MC013|               90|
|     MC014|               80|
|     MC015|               90|
|     MC016|               75|
|     MC017|               85|
|     MC018|               85|
|     MC019|               75|
|     MC020|               80|
+----------+-----------------+
only showing top 20 rows


In [136]:
machine_json.select(
    "machine_id",
    "specifications.vibration_limit_mm_s"
).show()

+----------+--------------------+
|machine_id|vibration_limit_mm_s|
+----------+--------------------+
|     MC001|                 5.0|
|     MC002|                 5.5|
|     MC003|                 4.5|
|     MC004|                 4.5|
|     MC005|                 4.5|
|     MC006|                 5.5|
|     MC007|                 4.5|
|     MC008|                 5.0|
|     MC009|                 5.5|
|     MC010|                 4.5|
|     MC011|                 6.0|
|     MC012|                 5.5|
|     MC013|                 5.0|
|     MC014|                 5.5|
|     MC015|                 4.5|
|     MC016|                 5.5|
|     MC017|                 4.5|
|     MC018|                 5.0|
|     MC019|                 5.0|
|     MC020|                 5.0|
+----------+--------------------+
only showing top 20 rows


In [138]:
machine_json.select(
    "machine_id",
    "maintenance"
).show()

+----------+----------------+
|machine_id|     maintenance|
+----------+----------------+
|     MC001|{60, 2026-01-14}|
|     MC002|{60, 2026-05-19}|
|     MC003|{45, 2026-08-28}|
|     MC004|{60, 2026-04-19}|
|     MC005|{30, 2026-05-09}|
|     MC006|{60, 2026-01-09}|
|     MC007|{30, 2026-06-26}|
|     MC008|{30, 2026-04-21}|
|     MC009|{30, 2026-08-13}|
|     MC010|{45, 2026-06-05}|
|     MC011|{30, 2026-04-04}|
|     MC012|{30, 2026-04-26}|
|     MC013|{60, 2026-08-03}|
|     MC014|{90, 2026-01-21}|
|     MC015|{30, 2026-09-02}|
|     MC016|{45, 2026-02-15}|
|     MC017|{30, 2026-06-23}|
|     MC018|{30, 2026-03-27}|
|     MC019|{30, 2026-07-27}|
|     MC020|{45, 2026-09-17}|
+----------+----------------+
only showing top 20 rows


In [139]:
machine_json.select(
    "machine_id",
    "sensors"
).show(truncate=False)

+----------+----------------------------------------------------------------------+
|machine_id|sensors                                                               |
+----------+----------------------------------------------------------------------+
|MC001     |[{true, temperature, C}, {true, vibration, mm/s}, {true, energy, kWh}]|
|MC002     |[{true, temperature, C}, {true, vibration, mm/s}, {true, energy, kWh}]|
|MC003     |[{true, temperature, C}, {true, vibration, mm/s}, {true, energy, kWh}]|
|MC004     |[{true, temperature, C}, {true, vibration, mm/s}, {true, energy, kWh}]|
|MC005     |[{true, temperature, C}, {true, vibration, mm/s}, {true, energy, kWh}]|
|MC006     |[{true, temperature, C}, {true, vibration, mm/s}, {true, energy, kWh}]|
|MC007     |[{true, temperature, C}, {true, vibration, mm/s}, {true, energy, kWh}]|
|MC008     |[{true, temperature, C}, {true, vibration, mm/s}, {true, energy, kWh}]|
|MC009     |[{true, temperature, C}, {true, vibration, mm/s}, {true, energy,

In [141]:
sensor_df = machine_json.withColumn(
    "sensor",
    F.explode("sensors")
)
sensor_df.show(truncate=False)

+-----------------+--------------+----------+------------+----------------+--------+----------------------------------------------------------------------+--------------+-----------------------+
|capacity_per_hour|installed_year|machine_id|machine_type|maintenance     |plant_id|sensors                                                               |specifications|sensor                 |
+-----------------+--------------+----------+------------+----------------+--------+----------------------------------------------------------------------+--------------+-----------------------+
|120              |2022          |MC001     |CNC         |{60, 2026-01-14}|PL01    |[{true, temperature, C}, {true, vibration, mm/s}, {true, energy, kWh}]|{85, 22, 5.0} |{true, temperature, C} |
|120              |2022          |MC001     |CNC         |{60, 2026-01-14}|PL01    |[{true, temperature, C}, {true, vibration, mm/s}, {true, energy, kWh}]|{85, 22, 5.0} |{true, vibration, mm/s}|
|120              |2022  

In [142]:
sensor_df.select(
    "machine_id",
    "sensor.*"
).show()

+----------+------+-----------+----+
|machine_id|active|     sensor|unit|
+----------+------+-----------+----+
|     MC001|  true|temperature|   C|
|     MC001|  true|  vibration|mm/s|
|     MC001|  true|     energy| kWh|
|     MC002|  true|temperature|   C|
|     MC002|  true|  vibration|mm/s|
|     MC002|  true|     energy| kWh|
|     MC003|  true|temperature|   C|
|     MC003|  true|  vibration|mm/s|
|     MC003|  true|     energy| kWh|
|     MC004|  true|temperature|   C|
|     MC004|  true|  vibration|mm/s|
|     MC004|  true|     energy| kWh|
|     MC005|  true|temperature|   C|
|     MC005|  true|  vibration|mm/s|
|     MC005|  true|     energy| kWh|
|     MC006|  true|temperature|   C|
|     MC006|  true|  vibration|mm/s|
|     MC006|  true|     energy| kWh|
|     MC007|  true|temperature|   C|
|     MC007|  true|  vibration|mm/s|
+----------+------+-----------+----+
only showing top 20 rows


In [145]:
sensor_df.filter(
    (F.col("sensor.unit") == "temperature_c") &
    (F.col("sensor.active") == True)
).select(
    "machine_id",
    "sensor"
).show(truncate=False)

+----------+------+
|machine_id|sensor|
+----------+------+
+----------+------+



In [146]:
json_joined_df = df.join(
    machine_json,
    on="machine_id",
    how="left"
)

json_joined_df.show()

+----------+-------------+---------------+--------+-----------+-------+------------+--------------+-------------+--------------+---------------+----------------+----------+-------------+--------------+--------------+----------------+----------+-----------------+--------------+------------+----------------+--------+--------------------+--------------+
|machine_id|production_id|production_date|plant_id|operator_id|  shift|product_code|product_family|units_planned|units_produced|defective_units|downtime_minutes|energy_kwh|temperature_c|vibration_mm_s|quality_status|maintenance_flag|supervisor|capacity_per_hour|installed_year|machine_type|     maintenance|plant_id|             sensors|specifications|
+----------+-------------+---------------+--------+-----------+-------+------------+--------------+-------------+--------------+---------------+----------------+----------+-------------+--------------+--------------+----------------+----------+-----------------+--------------+------------+----

16. Union and Union ALL

In [147]:
extra_df = spark.read.csv(
"manufacturing_production_extra_12k.csv",
    header=True,
    inferSchema=True
)

In [148]:
print(df.count())

120000


In [149]:
print(extra_df.count())

12500


In [150]:
extra_aligned = extra_df.select(df.columns)

In [151]:
union_df = df.union(extra_aligned)

print(union_df.count())

132500


In [152]:
union_all_df = df.unionAll(extra_aligned)

print(union_all_df.count())

132500


In [153]:
print("Main:", df.count())
print("Extra:", extra_df.count())
print("Union:", union_df.count())
print("Union All:", union_all_df.count())

Main: 120000
Extra: 12500
Union: 132500
Union All: 132500


In [154]:
union_df.groupBy("production_id").count().filter(
    F.col("count") > 1
).show()

+-------------+-----+
|production_id|count|
+-------------+-----+
|    PR0000413|    2|
|    PR0000135|    2|
|    PR0000236|    2|
|    PR0000339|    2|
|    PR0000059|    2|
|    PR0000463|    2|
|    PR0000136|    2|
|    PR0000263|    2|
|    PR0000393|    2|
|    PR0000445|    2|
|    PR0000124|    2|
|    PR0000299|    2|
|    PR0000341|    2|
|    PR0000008|    2|
|    PR0000172|    2|
|    PR0000359|    2|
|    PR0000389|    2|
|    PR0000026|    2|
|    PR0000274|    2|
|    PR0000161|    2|
+-------------+-----+
only showing top 20 rows


In [155]:
unique_df = union_df.dropDuplicates()

unique_df.show()

+-------------+---------------+--------+----------+-----------+-------+------------+--------------+-------------+--------------+---------------+----------------+----------+-------------+--------------+--------------+----------------+----------+
|production_id|production_date|plant_id|machine_id|operator_id|  shift|product_code|product_family|units_planned|units_produced|defective_units|downtime_minutes|energy_kwh|temperature_c|vibration_mm_s|quality_status|maintenance_flag|supervisor|
+-------------+---------------+--------+----------+-----------+-------+------------+--------------+-------------+--------------+---------------+----------------+----------+-------------+--------------+--------------+----------------+----------+
|    PR0000304|     2026-07-31|    PL01|     MC004|     OP0518|Evening|     MTR-200|        Motors|          339|           288|              0|              57|     83.44|        58.53|          1.47|          Pass|              No|    Bharat|
|    PR0000408|     

In [156]:
print("Before:", union_df.count())
print("After:", unique_df.count())

Before: 132500
After: 132500


In [157]:
union_by_name = df.unionByName(extra_df)

union_by_name.show()

+-------------+---------------+--------+----------+-----------+-------+------------+--------------+-------------+--------------+---------------+----------------+----------+-------------+--------------+--------------+----------------+----------+
|production_id|production_date|plant_id|machine_id|operator_id|  shift|product_code|product_family|units_planned|units_produced|defective_units|downtime_minutes|energy_kwh|temperature_c|vibration_mm_s|quality_status|maintenance_flag|supervisor|
+-------------+---------------+--------+----------+-----------+-------+------------+--------------+-------------+--------------+---------------+----------------+----------+-------------+--------------+--------------+----------------+----------+
|    PR0000001|     2026-08-07|    PL03|     MC011|     OP0275|Evening|     MTR-100|        Motors|         1060|           993|             30|              34|    388.11|         61.4|          3.88|        Rework|              No|    Charan|
|    PR0000002|     

In [158]:
rearranged_df = extra_df.select(
    *reversed(extra_df.columns)
)

final_union = df.unionByName(rearranged_df)

final_union.show()

+-------------+---------------+--------+----------+-----------+-------+------------+--------------+-------------+--------------+---------------+----------------+----------+-------------+--------------+--------------+----------------+----------+
|production_id|production_date|plant_id|machine_id|operator_id|  shift|product_code|product_family|units_planned|units_produced|defective_units|downtime_minutes|energy_kwh|temperature_c|vibration_mm_s|quality_status|maintenance_flag|supervisor|
+-------------+---------------+--------+----------+-----------+-------+------------+--------------+-------------+--------------+---------------+----------------+----------+-------------+--------------+--------------+----------------+----------+
|    PR0000001|     2026-08-07|    PL03|     MC011|     OP0275|Evening|     MTR-100|        Motors|         1060|           993|             30|              34|    388.11|         61.4|          3.88|        Rework|              No|    Charan|
|    PR0000002|     

17. Window Functions


In [159]:
from pyspark.sql.window import Window

w = Window.partitionBy("plant_id").orderBy(
    F.col("units_produced").desc()
)

In [160]:
rank_df = df.withColumn(
    "row_number",
    F.row_number().over(w)
).withColumn(
    "rank",
    F.rank().over(w)
).withColumn(
    "dense_rank",
    F.dense_rank().over(w)
)

rank_df.select(
    "plant_id",
    "machine_id",
    "units_produced",
    "row_number",
    "rank",
    "dense_rank"
).show()


+--------+----------+--------------+----------+----+----------+
|plant_id|machine_id|units_produced|row_number|rank|dense_rank|
+--------+----------+--------------+----------+----+----------+
|    PL01|     MC003|          1239|         1|   1|         1|
|    PL01|     MC005|          1238|         2|   2|         2|
|    PL01|     MC002|          1236|         3|   3|         3|
|    PL01|     MC002|          1236|         4|   3|         3|
|    PL01|     MC002|          1236|         5|   3|         3|
|    PL01|     MC004|          1234|         6|   6|         4|
|    PL01|     MC005|          1233|         7|   7|         5|
|    PL01|     MC002|          1230|         8|   8|         6|
|    PL01|     MC002|          1229|         9|   9|         7|
|    PL01|     MC003|          1229|        10|   9|         7|
|    PL01|     MC002|          1229|        11|   9|         7|
|    PL01|     MC005|          1228|        12|  12|         8|
|    PL01|     MC005|          1228|    

In [161]:
rank_df.filter(
    F.col("row_number") <= 5
).show()

+-------------+---------------+--------+----------+-----------+-------+------------+--------------+-------------+--------------+---------------+----------------+----------+-------------+--------------+--------------+----------------+----------+----------+----+----------+
|production_id|production_date|plant_id|machine_id|operator_id|  shift|product_code|product_family|units_planned|units_produced|defective_units|downtime_minutes|energy_kwh|temperature_c|vibration_mm_s|quality_status|maintenance_flag|supervisor|row_number|rank|dense_rank|
+-------------+---------------+--------+----------+-----------+-------+------------+--------------+-------------+--------------+---------------+----------------+----------+-------------+--------------+--------------+----------------+----------+----------+----+----------+
|    PR0073059|     2026-04-01|    PL01|     MC003|     OP0110|Evening|     MTR-200|        Motors|         1198|          1239|             11|               0|     282.6|        50.3

In [162]:
machine_output = df.groupBy(
    "plant_id",
    "machine_id"
).agg(
    F.sum("units_produced").alias("total_output")
)

machine_window = Window.partitionBy(
    "plant_id"
).orderBy(
    F.col("total_output").desc(),
    F.col("machine_id")
)

machine_output.withColumn(
    "rn",
    F.row_number().over(machine_window)
).filter(
    F.col("rn") <= 3
).show()

+--------+----------+------------+---+
|plant_id|machine_id|total_output| rn|
+--------+----------+------------+---+
|    PL01|     MC004|     2722412|  1|
|    PL01|     MC002|     2713779|  2|
|    PL01|     MC001|     2691431|  3|
|    PL02|     MC009|     2760070|  1|
|    PL02|     MC007|     2713038|  2|
|    PL02|     MC008|     2705850|  3|
|    PL03|     MC012|     2750570|  1|
|    PL03|     MC015|     2739284|  2|
|    PL03|     MC014|     2735500|  3|
|    PL04|     MC016|     2750780|  1|
|    PL04|     MC019|     2735839|  2|
|    PL04|     MC017|     2726300|  3|
|    PL05|     MC024|     2737331|  1|
|    PL05|     MC023|     2718075|  2|
|    PL05|     MC022|     2635629|  3|
|    PL06|     MC026|     2723306|  1|
|    PL06|     MC027|     2718877|  2|
|    PL06|     MC030|     2701993|  3|
+--------+----------+------------+---+



In [163]:
plant_window = Window.partitionBy("plant_id")

df.withColumn(
    "plant_total",
    F.sum("units_produced").over(plant_window)
).show()

+-------------+---------------+--------+----------+-----------+-------+------------+--------------+-------------+--------------+---------------+----------------+----------+-------------+--------------+--------------+----------------+----------+-----------+
|production_id|production_date|plant_id|machine_id|operator_id|  shift|product_code|product_family|units_planned|units_produced|defective_units|downtime_minutes|energy_kwh|temperature_c|vibration_mm_s|quality_status|maintenance_flag|supervisor|plant_total|
+-------------+---------------+--------+----------+-----------+-------+------------+--------------+-------------+--------------+---------------+----------------+----------+-------------+--------------+--------------+----------------+----------+-----------+
|    PR0000007|     2026-05-07|    PL02|     MC007|     OP0645|Morning|     VLV-120|        Valves|         1165|           955|             10|               5|    355.56|        62.76|          2.89|          Pass|             

In [164]:
machine_avg_window = Window.partitionBy("machine_id")

df.withColumn(
    "machine_avg",
    F.avg("units_produced").over(machine_avg_window)
).show()


+-------------+---------------+--------+----------+-----------+-------+------------+--------------+-------------+--------------+---------------+----------------+----------+-------------+--------------+--------------+----------------+----------+-----------------+
|production_id|production_date|plant_id|machine_id|operator_id|  shift|product_code|product_family|units_planned|units_produced|defective_units|downtime_minutes|energy_kwh|temperature_c|vibration_mm_s|quality_status|maintenance_flag|supervisor|      machine_avg|
+-------------+---------------+--------+----------+-----------+-------+------------+--------------+-------------+--------------+---------------+----------------+----------+-------------+--------------+--------------+----------------+----------+-----------------+
|    PR0000003|     2026-05-20|    PL01|     MC002|     OP0189|Evening|     MTR-300|        Motors|          442|           380|             17|              15|    183.34|        54.13|          4.66|        Re

In [165]:
date_window = Window.partitionBy(
    "machine_id"
).orderBy(
    "production_date",
    "production_id"
).rowsBetween(
    Window.unboundedPreceding,
    Window.currentRow
)

df.withColumn(
    "cumulative_production",
    F.sum("units_produced").over(date_window)
).show()


+-------------+---------------+--------+----------+-----------+-------+------------+--------------+-------------+--------------+---------------+----------------+----------+-------------+--------------+--------------+----------------+----------+---------------------+
|production_id|production_date|plant_id|machine_id|operator_id|  shift|product_code|product_family|units_planned|units_produced|defective_units|downtime_minutes|energy_kwh|temperature_c|vibration_mm_s|quality_status|maintenance_flag|supervisor|cumulative_production|
+-------------+---------------+--------+----------+-----------+-------+------------+--------------+-------------+--------------+---------------+----------------+----------+-------------+--------------+--------------+----------------+----------+---------------------+
|    PR0001004|     2026-01-01|    PL01|     MC002|     OP0424|  Night|     VLV-120|        Valves|          517|           520|             14|               0|    133.77|        52.65|          4.5

In [166]:
previous_window = Window.partitionBy(
    "machine_id"
).orderBy(
    "production_date",
    "production_id"
)

previous_df = df.withColumn(
    "previous_output",
    F.lag("units_produced", 1).over(previous_window)
)

previous_df.show()

+-------------+---------------+--------+----------+-----------+-------+------------+--------------+-------------+--------------+---------------+----------------+----------+-------------+--------------+--------------+----------------+----------+---------------+
|production_id|production_date|plant_id|machine_id|operator_id|  shift|product_code|product_family|units_planned|units_produced|defective_units|downtime_minutes|energy_kwh|temperature_c|vibration_mm_s|quality_status|maintenance_flag|supervisor|previous_output|
+-------------+---------------+--------+----------+-----------+-------+------------+--------------+-------------+--------------+---------------+----------------+----------+-------------+--------------+--------------+----------------+----------+---------------+
|    PR0001004|     2026-01-01|    PL01|     MC002|     OP0424|  Night|     VLV-120|        Valves|          517|           520|             14|               0|    133.77|        52.65|          4.53|          Pass| 

In [167]:
next_df = df.withColumn(
    "next_output",
    F.lead("units_produced", 1).over(previous_window)
)

next_df.show()

+-------------+---------------+--------+----------+-----------+-------+------------+--------------+-------------+--------------+---------------+----------------+----------+-------------+--------------+--------------+----------------+----------+-----------+
|production_id|production_date|plant_id|machine_id|operator_id|  shift|product_code|product_family|units_planned|units_produced|defective_units|downtime_minutes|energy_kwh|temperature_c|vibration_mm_s|quality_status|maintenance_flag|supervisor|next_output|
+-------------+---------------+--------+----------+-----------+-------+------------+--------------+-------------+--------------+---------------+----------------+----------+-------------+--------------+--------------+----------------+----------+-----------+
|    PR0001004|     2026-01-01|    PL01|     MC002|     OP0424|  Night|     VLV-120|        Valves|          517|           520|             14|               0|    133.77|        52.65|          4.53|          Pass|             

In [168]:
previous_df.withColumn(
    "output_change",
    F.col("units_produced") - F.col("previous_output")
).show()

+-------------+---------------+--------+----------+-----------+-------+------------+--------------+-------------+--------------+---------------+----------------+----------+-------------+--------------+--------------+----------------+----------+---------------+-------------+
|production_id|production_date|plant_id|machine_id|operator_id|  shift|product_code|product_family|units_planned|units_produced|defective_units|downtime_minutes|energy_kwh|temperature_c|vibration_mm_s|quality_status|maintenance_flag|supervisor|previous_output|output_change|
+-------------+---------------+--------+----------+-----------+-------+------------+--------------+-------------+--------------+---------------+----------------+----------+-------------+--------------+--------------+----------------+----------+---------------+-------------+
|    PR0001004|     2026-01-01|    PL01|     MC002|     OP0424|  Night|     VLV-120|        Valves|          517|           520|             14|               0|    133.77|   

In [169]:
family_df = df.groupBy("product_family").agg(
    F.sum("units_produced").alias("total_output")
)

family_window = Window.orderBy(
    F.col("total_output").desc()
)

family_df.withColumn(
    "family_rank",
    F.rank().over(family_window)
).show()

+--------------+------------+-----------+
|product_family|total_output|family_rank|
+--------------+------------+-----------+
|        Valves|    20433082|          1|
|         Pumps|    20208739|          2|
|   Controllers|    20168572|          3|
|        Motors|    20117495|          4|
+--------------+------------+-----------+



18. Date and Time Functions

In [170]:
date_df = df.withColumn(
    "production_date",
    F.to_date("production_date")
)

In [171]:
date_df = date_df.withColumn(
    "year",
    F.year("production_date")
)

In [172]:
date_df = date_df.withColumn(
    "month",
    F.month("production_date")
)

In [173]:
date_df = date_df.withColumn(
    "day",
    F.dayofmonth("production_date")
)

In [174]:
date_df = date_df.withColumn(
    "month_name",
    F.date_format("production_date", "MMMM")
)

In [175]:
date_df = date_df.withColumn(
    "day_name",
    F.date_format("production_date", "EEEE")
)

In [176]:
date_df = date_df.withColumn(
    "year_month",
    F.date_format("production_date", "yyyy-MM")
)

date_df.show()

+-------------+---------------+--------+----------+-----------+-------+------------+--------------+-------------+--------------+---------------+----------------+----------+-------------+--------------+--------------+----------------+----------+----+-----+---+----------+---------+----------+
|production_id|production_date|plant_id|machine_id|operator_id|  shift|product_code|product_family|units_planned|units_produced|defective_units|downtime_minutes|energy_kwh|temperature_c|vibration_mm_s|quality_status|maintenance_flag|supervisor|year|month|day|month_name| day_name|year_month|
+-------------+---------------+--------+----------+-----------+-------+------------+--------------+-------------+--------------+---------------+----------------+----------+-------------+--------------+--------------+----------------+----------+----+-----+---+----------+---------+----------+
|    PR0000001|     2026-08-07|    PL03|     MC011|     OP0275|Evening|     MTR-100|        Motors|         1060|           

In [177]:
monthly_production = date_df.groupBy(
    "year_month"
).agg(
    F.sum("units_produced").alias("total_production")
).orderBy("year_month")

monthly_production.show()

+----------+----------------+
|year_month|total_production|
+----------+----------------+
|   2026-01|         8478059|
|   2026-02|         7810891|
|   2026-03|         8441121|
|   2026-04|         8211005|
|   2026-05|         8507542|
|   2026-06|         8400505|
|   2026-07|         8476416|
|   2026-08|         8529150|
|   2026-09|         8115221|
|   2026-10|         5957978|
+----------+----------------+



In [178]:
date_df.groupBy("year_month").agg(
    F.sum("defective_units").alias("total_defects")
).orderBy("year_month").show()

+----------+-------------+
|year_month|total_defects|
+----------+-------------+
|   2026-01|       211741|
|   2026-02|       197462|
|   2026-03|       212036|
|   2026-04|       208707|
|   2026-05|       212049|
|   2026-06|       210381|
|   2026-07|       210482|
|   2026-08|       214807|
|   2026-09|       202990|
|   2026-10|       149589|
+----------+-------------+



In [179]:
date_df.groupBy(
    "year_month",
    "plant_id"
).agg(
    F.sum("units_produced").alias("total")
).orderBy("year_month", "plant_id").show()

+----------+--------+-------+
|year_month|plant_id|  total|
+----------+--------+-------+
|   2026-01|    PL01|1451436|
|   2026-01|    PL02|1419252|
|   2026-01|    PL03|1429811|
|   2026-01|    PL04|1399488|
|   2026-01|    PL05|1375887|
|   2026-01|    PL06|1402185|
|   2026-02|    PL01|1342273|
|   2026-02|    PL02|1278581|
|   2026-02|    PL03|1305531|
|   2026-02|    PL04|1292557|
|   2026-02|    PL05|1324271|
|   2026-02|    PL06|1267678|
|   2026-03|    PL01|1442140|
|   2026-03|    PL02|1359740|
|   2026-03|    PL03|1416484|
|   2026-03|    PL04|1447603|
|   2026-03|    PL05|1411435|
|   2026-03|    PL06|1363719|
|   2026-04|    PL01|1350450|
|   2026-04|    PL02|1369684|
+----------+--------+-------+
only showing top 20 rows


In [180]:
date_df.filter(
    F.col("production_date").between(
        "2026-01-01",
        "2026-06-30"
    )
).show()

+-------------+---------------+--------+----------+-----------+-------+------------+--------------+-------------+--------------+---------------+----------------+----------+-------------+--------------+--------------+----------------+----------+----+-----+---+----------+---------+----------+
|production_id|production_date|plant_id|machine_id|operator_id|  shift|product_code|product_family|units_planned|units_produced|defective_units|downtime_minutes|energy_kwh|temperature_c|vibration_mm_s|quality_status|maintenance_flag|supervisor|year|month|day|month_name| day_name|year_month|
+-------------+---------------+--------+----------+-----------+-------+------------+--------------+-------------+--------------+---------------+----------------+----------+-------------+--------------+--------------+----------------+----------+----+-----+---+----------+---------+----------+
|    PR0000003|     2026-05-20|    PL01|     MC002|     OP0189|Evening|     MTR-300|        Motors|          442|           

In [181]:
date_df.withColumn(
    "after_7_days",
    F.date_add("production_date", 7)
).show()

+-------------+---------------+--------+----------+-----------+-------+------------+--------------+-------------+--------------+---------------+----------------+----------+-------------+--------------+--------------+----------------+----------+----+-----+---+----------+---------+----------+------------+
|production_id|production_date|plant_id|machine_id|operator_id|  shift|product_code|product_family|units_planned|units_produced|defective_units|downtime_minutes|energy_kwh|temperature_c|vibration_mm_s|quality_status|maintenance_flag|supervisor|year|month|day|month_name| day_name|year_month|after_7_days|
+-------------+---------------+--------+----------+-----------+-------+------------+--------------+-------------+--------------+---------------+----------------+----------+-------------+--------------+--------------+----------------+----------+----+-----+---+----------+---------+----------+------------+
|    PR0000001|     2026-08-07|    PL03|     MC011|     OP0275|Evening|     MTR-100| 

In [182]:
date_df.withColumn(
    "before_30_days",
    F.date_sub("production_date", 30)
).show()

+-------------+---------------+--------+----------+-----------+-------+------------+--------------+-------------+--------------+---------------+----------------+----------+-------------+--------------+--------------+----------------+----------+----+-----+---+----------+---------+----------+--------------+
|production_id|production_date|plant_id|machine_id|operator_id|  shift|product_code|product_family|units_planned|units_produced|defective_units|downtime_minutes|energy_kwh|temperature_c|vibration_mm_s|quality_status|maintenance_flag|supervisor|year|month|day|month_name| day_name|year_month|before_30_days|
+-------------+---------------+--------+----------+-----------+-------+------------+--------------+-------------+--------------+---------------+----------------+----------+-------------+--------------+--------------+----------------+----------+----+-----+---+----------+---------+----------+--------------+
|    PR0000001|     2026-08-07|    PL03|     MC011|     OP0275|Evening|     MTR

In [183]:
date_df.withColumn(
    "days_difference",
    F.datediff(
        F.current_date(),
        F.col("production_date")
    )
).show()

+-------------+---------------+--------+----------+-----------+-------+------------+--------------+-------------+--------------+---------------+----------------+----------+-------------+--------------+--------------+----------------+----------+----+-----+---+----------+---------+----------+---------------+
|production_id|production_date|plant_id|machine_id|operator_id|  shift|product_code|product_family|units_planned|units_produced|defective_units|downtime_minutes|energy_kwh|temperature_c|vibration_mm_s|quality_status|maintenance_flag|supervisor|year|month|day|month_name| day_name|year_month|days_difference|
+-------------+---------------+--------+----------+-----------+-------+------------+--------------+-------------+--------------+---------------+----------------+----------+-------------+--------------+--------------+----------------+----------+----+-----+---+----------+---------+----------+---------------+
|    PR0000001|     2026-08-07|    PL03|     MC011|     OP0275|Evening|     

In [184]:
monthly_production.orderBy(
    F.col("total_production").desc()
).show(1)

+----------+----------------+
|year_month|total_production|
+----------+----------------+
|   2026-08|         8529150|
+----------+----------------+
only showing top 1 row
